# ForestWatch Papua — Banding 3 Model, Tetapkan Pemenang & Evaluasi Mendalam

**Jalankan di SATU komputer SETELAH ke-3 notebook training selesai & Drive ter-sync.**

Notebook ini **fokus banding + pilih model + evaluasi** (inferensi T1/T2 dan generate 7 file kontrak **DITUNDA** ke notebook terpisah nanti -- IoU ketiga model belum cukup baik untuk dipakai produksi).

Notebook ini: (1) baca `summary.json` + `metrics.json` ke-3 model dari `ForestWatch_Outputs/Model_Comparison/`, (2) bandingkan akurasi (test mIoU + FWIoU pada Papua holdout yang identik), (3) tabel IoU per-kelas LINTAS 3 model -- cari model mana yang unggul di kelas apa, (4) Producer's/User's Accuracy (recall/precision) per kelas untuk diagnosis arah error, (5) analisis plafon ensemble (batas atas teoretis kombinasi 3 model) sebagai dasar keputusan combine-model nanti, (6) tetapkan **pemenang** + promosikan artefaknya ke lokasi kanonik (`ForestWatch_Patches/best_model.pt`, `ForestWatch_Outputs/`).


## Bagian 0 — Setup environment (Colab **atau** Komputer Lab)

Notebook ini **berdiri sendiri**. Ia **memuat hasil EDA & preprocessing**
(Bagian 1–14 dari `forestwatch_papua_full_pipeline.ipynb`) yang sudah tersimpan
di Google Drive — **tidak menghitung ulang**.

- **Google Colab** → set `ENV = "colab"`. Sel setup meng-clone repo, install
  package, lalu mount Drive.
- **Komputer lab** → set `ENV = "lab"`. Prasyarat **sekali saja**:
  1. Install **Google Drive for Desktop**, login akun yang sama, set folder
     `Satria Data 3.0` ke mode **Mirror** (bukan *Stream-only*) supaya file `.npz`
     benar-benar ada di disk lokal (DataLoader membaca ribuan file tiap epoch).
  2. Di clone repo lokal jalankan: `pip install -e ".[ml]"`.
  3. Sesuaikan `DRIVE_ROOT` ke path mount Drive Desktop (mis. `G:/My Drive/Satria Data 3.0`).

In [ ]:
# === Bagian 0 — Setup (set ENV = "colab" ATAU "lab") ===
ENV = "lab"   # ganti "colab" kalau jalan di Google Colab
from pathlib import Path

if ENV == "colab":
    import subprocess, sys, importlib
    # clone pertama kali / pull sesi berikutnya (selalu kode terbaru), lalu install.
    subprocess.run(
        "cd /content && (git -C fw_repo pull -q || git clone --depth 1 "
        "https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo)",
        shell=True, check=False,
    )
    subprocess.run("pip install -q -e /content/fw_repo[gee,gis,ml]", shell=True, check=False)
    if "/content/fw_repo/src" not in sys.path:
        sys.path.insert(0, "/content/fw_repo/src")
    for _m in [m for m in list(sys.modules) if m == "forestwatch" or m.startswith("forestwatch.")]:
        del sys.modules[_m]
    importlib.invalidate_caches()
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/Satria Data 3.0")
elif ENV == "lab":
    DRIVE_ROOT = Path(r"G:/My Drive/Satria Data 3.0")   # <- SESUAIKAN path mount Drive Desktop lab
else:
    raise ValueError("ENV harus 'colab' atau 'lab'")

assert DRIVE_ROOT.exists(), (
    f"DRIVE_ROOT {DRIVE_ROOT} tidak ada — cek mount Drive / sync (mode Mirror)."
)

import torch
_gpu = f" ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else ""
print(f"ENV={ENV} | DRIVE_ROOT={DRIVE_ROOT} | CUDA={torch.cuda.is_available()}{_gpu}")

In [ ]:
# === Deklarasi path gdrive (hasil EDA & preprocessing tersimpan di sini) ===
TILES_T1   = DRIVE_ROOT / 'ForestWatch_Tiles_T1'
TILES_T2   = DRIVE_ROOT / 'ForestWatch_Tiles_T2'
PATCH_DIR  = DRIVE_ROOT / 'ForestWatch_Patches'           # patch Papua (+ ckpt kanonik)
PATCHES_TRANSFER  = DRIVE_ROOT / 'ForestWatch_Patches_Transfer'
AUGMENTED_PATCHES = DRIVE_ROOT / 'Augmented_Patches'
DIST_DIR   = DRIVE_ROOT / 'Distribution_Reports'
MASK_DIR   = DRIVE_ROOT / 'ForestWatch_Masks'
OUT_DIR    = DRIVE_ROOT / 'ForestWatch_Outputs'
MODELS_ROOT = OUT_DIR / 'Model_Comparison'                # folder induk 3 model
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

from forestwatch.config import load_config
from forestwatch.utils.io import save_json, load_json
cfg = load_config()
print('Resep training :', cfg['training']['loss']['type'],
      '| epochs:', cfg['training']['epochs'], '| batch:', cfg['training']['batch_size'])
print('MODELS_ROOT    :', MODELS_ROOT)

## Bagian 15.5 — Banding 3 Model & Tetapkan Pemenang

In [ ]:
# === Bagian 15.5 — Baca summary.json + metrics.json ke-3 model + tabel + bar chart ===
import json
import numpy as np
import matplotlib.pyplot as plt
from forestwatch.utils.io import save_json, load_json
from forestwatch.training.metrics import frequency_weighted_iou

EXPECTED = ['model_1_attention_unet', 'model_2_deeplabv3plus', 'model_3_unetpp']
rows = []
cms = {}  # model_key -> confusion matrix (np.ndarray), dipakai di analisis lanjut
for key in EXPECTED:
    p_summary = MODELS_ROOT / key / 'summary.json'
    p_metrics = MODELS_ROOT / key / 'metrics.json'
    if p_summary.exists() and p_metrics.exists():
        row = json.load(open(p_summary))
        cm = np.array(json.load(open(p_metrics))['confusion_matrix'])
        row['fwiou'] = frequency_weighted_iou(cm)
        rows.append(row)
        cms[key] = cm
    else:
        print(f"[belum ada] {key} — summary.json/metrics.json belum lengkap / belum sync.")
assert rows, "Belum ada summary.json+metrics.json satu pun. Jalankan notebook training dulu."

print(f"\n{'model':<26}{'arch':<15}{'val mIoU':>9}{'test mIoU':>10}{'FWIoU':>8}{'OA':>8}{'kappa':>8}{'param':>13}{'menit':>7}")
print('-' * 104)
for r in sorted(rows, key=lambda x: x['test_mean_iou'], reverse=True):
    print(f"{r['model_key']:<26}{r['architecture']:<15}{r['best_val_iou']:>9.4f}{r['test_mean_iou']:>10.4f}"
          f"{r['fwiou']:>8.4f}{r['test_overall_accuracy']*100:>7.1f}%{r['test_kappa']:>8.4f}{r['n_parameters']:>13,}{r['train_minutes']:>7}")

fig, ax = plt.subplots(figsize=(8, 4))
names = [r['model_key'] for r in rows]; mious = [r['test_mean_iou'] for r in rows]
ax.bar(names, mious, color=['#2c7fb8', '#7fcdbb', '#c7e9b4'][:len(rows)])
for i, v in enumerate(mious):
    ax.text(i, v, f'{v:.3f}', ha='center', va='bottom')
ax.set_ylabel('Test mIoU (Papua holdout)'); ax.set_ylim(0, 1)
ax.set_title('Perbandingan 3 Model — Test mIoU')
plt.xticks(rotation=12, ha='right'); fig.tight_layout()
fig.savefig(MODELS_ROOT / 'comparison.png', dpi=120, bbox_inches='tight'); plt.show()
save_json({'models': rows}, MODELS_ROOT / 'comparison.json')
print('Disimpan:', MODELS_ROOT / 'comparison.png', '+ comparison.json')


## Bagian 15.6 — IoU per-kelas LINTAS 3 model

Tabel ini menjawab langsung: **model mana unggul di kelas apa**. Ini bukti komplementaritas (atau tidak) antar-arsitektur, dan dasar paling penting sebelum memutuskan apakah kombinasi/ensemble 2 model layak dikejar.


In [ ]:
# === Tabel IoU per-kelas lintas-model + juara per kelas ===
from forestwatch.constants import CLASS_NAMES
from forestwatch.training.metrics import compute_iou_per_class

model_keys = [r['model_key'] for r in rows]
iou_per_model = {k: compute_iou_per_class(cms[k]) for k in model_keys}

col_w = 18
print(f"{'kelas':<16}" + ''.join(f"{k:>{col_w}}" for k in model_keys) + f"{'juara':>16}")
print('-' * (16 + col_w * len(model_keys) + 16))
for c, cname in enumerate(CLASS_NAMES):
    ious = {k: float(iou_per_model[k][c]) for k in model_keys}
    champ = max(ious, key=ious.get)
    row_str = f"{cname:<16}" + ''.join(f"{ious[k]:>{col_w}.4f}" for k in model_keys)
    row_str += f"{champ:>16}"
    print(row_str)


## Bagian 15.7 — Producer's Accuracy (recall) & User's Accuracy (precision) per kelas

Istilah standar accuracy-assessment remote sensing (Olofsson dkk. 2014). **Producer's Accuracy (recall)** rendah = model banyak KELEWATAN kelas itu (FN tinggi). **User's Accuracy (precision)** rendah = model banyak SALAH NUDUH kelas itu (FP tinggi). Dua angka ini menjelaskan ARAH error di balik satu angka IoU.


In [ ]:
# === Producer's/User's Accuracy (recall/precision) per kelas, per model ===
from forestwatch.training.metrics import precision_per_class, recall_per_class

for k in model_keys:
    cm = cms[k]
    rec = recall_per_class(cm)
    prec = precision_per_class(cm)
    print(f"\n{k}")
    print(f"  {'kelas':<16}{'recall(PA)':>12}{'precision(UA)':>15}")
    for c, cname in enumerate(CLASS_NAMES):
        print(f"  {cname:<16}{rec[c]:>12.4f}{prec[c]:>15.4f}")


## Bagian 15.8 — Potensi ensemble: plafon mIoU oracle

`mIoU_oracle` = rata-rata, per kelas, dari IoU TERBAIK di antara 3 model (seakan ada "oracle" yang selalu pilih model terbaik per kelas per piksel). Ini **batas atas teoretis/optimistik, BUKAN jaminan capaian** -- ensemble nyata (soft-voting/logit-averaging atau routing per-kelas) harus diukur empiris di tahap terpisah, karena butuh prediksi mentah ke-3 model pada test set yang sama (belum tersimpan sekarang). Tapi kalau gap-nya besar, ini sinyal kuat: **kombinasi 2/3 model layak DIKEJAR**, bukan dilewatkan.


In [ ]:
# === Plafon ensemble (oracle ceiling) -- sinyal kelayakan combine-model ===
best_single_miou = max(r['test_mean_iou'] for r in rows)
best_single_key = max(rows, key=lambda x: x['test_mean_iou'])['model_key']

oracle_per_class = {
    cname: max(float(iou_per_model[k][c]) for k in model_keys)
    for c, cname in enumerate(CLASS_NAMES)
}
miou_oracle = sum(oracle_per_class.values()) / len(oracle_per_class)

print(f"Best single model : {best_single_key} -> mIoU={best_single_miou:.4f}")
print(f"mIoU oracle (plafon teoretis 3-model): {miou_oracle:.4f}")
print(f"Potensi gain jika ensemble efektif   : +{miou_oracle - best_single_miou:.4f}")
print()
print(f"{'kelas':<16}{'best-single':>12}{'oracle':>10}{'gap':>10}")
for c, cname in enumerate(CLASS_NAMES):
    best_single_c = float(iou_per_model[best_single_key][c])
    gap = oracle_per_class[cname] - best_single_c
    print(f"{cname:<16}{best_single_c:>12.4f}{oracle_per_class[cname]:>10.4f}{gap:>10.4f}")

if miou_oracle - best_single_miou > 0.03:
    print("\n=> Gap signifikan -- ensemble/combine-model PATUT DIKEJAR di tahap lanjut "
          "(butuh prediksi mentah ke-3 model di test set yang sama).")
else:
    print("\n=> Gap kecil -- ke-3 model relatif redundan per-kelas; ensemble mungkin "
          "tidak banyak menambah dibanding effort lapis lain (cRT/label-fix).")


In [ ]:
# === Tetapkan pemenang (test mIoU tertinggi) + promosikan ke lokasi kanonik ===
import shutil

winner = max(rows, key=lambda x: x['test_mean_iou'])
WIN_DIR = MODELS_ROOT / winner['model_key']
print(f"PEMENANG: {winner['model_key']} ({winner['architecture']}) — test mIoU={winner['test_mean_iou']:.4f}")
save_json(winner, MODELS_ROOT / 'winner.json')

# Promosikan artefak pemenang -> lokasi kanonik (supaya hilir/Orang-2 tak berubah).
CKPT_PATH = PATCH_DIR / 'best_model.pt'
OUT_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(WIN_DIR / 'best_model.pt', CKPT_PATH)
shutil.copy(WIN_DIR / 'metrics.json', OUT_DIR / 'metrics.json')
shutil.copy(WIN_DIR / 'model.onnx',  OUT_DIR / 'model.onnx')
shutil.copy(WIN_DIR / 'output' / 'confusion_matrix.png', OUT_DIR / 'confusion_matrix.png')
# selaraskan cfg ke arsitektur pemenang (utk model card).
cfg['model']['architecture'] = winner['architecture']
cfg['model']['encoder_name'] = winner['encoder_name']
print('Dipromosikan ->', CKPT_PATH)
print('             ->', OUT_DIR / 'metrics.json', '| model.onnx | confusion_matrix.png')
